# 03 — ruBERT-tiny2 fine-tune

`cointegrated/rubert-tiny2` is a compact (~29M parameters) Russian BERT
distilled for CPU inference. It is the best quality/compute trade-off for
this task without a GPU. We train for 3 epochs with class-weighted
cross-entropy to counter the severe class imbalance.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(root))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(context="talk", style="whitegrid")
RANDOM_STATE = 0

In [ ]:
tr = pd.read_parquet("../data/train_split.parquet")
va = pd.read_parquet("../data/val_split.parquet")
te = pd.read_parquet("../data/test_full.parquet")

# Downsample train only if you need speed; 48k on CPU with tiny2 is ~30-50 min / epoch.
SUBSAMPLE = 6000   # Set to None for the full 48k.  6k + 1 epoch finishes in ~10 min on CPU.
if SUBSAMPLE: tr = tr.sample(SUBSAMPLE, random_state=0).reset_index(drop=True)
len(tr), len(va), len(te)

In [ ]:
from src.models import fine_tune_cls, predict_proba_ft, FTConfig
cfg = FTConfig(model_name="cointegrated/rubert-tiny2", lr=3e-5, epochs=1, batch_size=32, max_len=128)  # bump to 3 epochs for the final run
mdl, tok = fine_tune_cls(tr, va, cfg)

In [ ]:
import torch
torch.save(mdl.state_dict(), "../results/rubert_tiny2_ft.pt")

## Validation metrics

In [ ]:
from src.eval import summarize, dump_metrics, report_text
from src.models import expected_round
proba_va = predict_proba_ft(mdl, tok, va["text"].tolist(), max_len=128, batch_size=64)
yp_argmax = proba_va.argmax(1)
yp_eround = np.clip(np.rint(proba_va @ np.arange(5)).astype(int), 0, 4)

m1 = summarize("B4_rubert_tiny_argmax", va["label"], yp_argmax)
m2 = summarize("B4_rubert_tiny_eround", va["label"], yp_eround)
print("argmax:", m1); print("E-round:", m2)
dump_metrics({"B4_rubert_tiny_argmax": m1, "B4_rubert_tiny_eround": m2})
print(report_text(va["label"], yp_argmax))

## Save probabilities for stacking in notebook 04

In [ ]:
proba_te = predict_proba_ft(mdl, tok, te["text"].tolist(), max_len=128, batch_size=64)
np.save("../data/proba_tiny_va.npy", proba_va); np.save("../data/proba_tiny_te.npy", proba_te)
print("saved", proba_va.shape, proba_te.shape)

## Submission from argmax

In [ ]:
from src.data import write_submission, IDX2LABEL
pred_rate = np.array([IDX2LABEL[i] for i in proba_te.argmax(1)])
print(write_submission(te["index"].values, pred_rate, "submission_B4_tiny_argmax.csv"))